# 04 — Ask the published Fabric data agent
This uses the documented MCP endpoint, not an authoring endpoint. Set `FABRIC_WORKSPACE_ID` and `FABRIC_DATA_AGENT_ID` in the notebook environment.

In [ ]:
%pip install -q mcp

In [ ]:
import asyncio, os
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

workspace_id = os.environ['FABRIC_WORKSPACE_ID']
agent_id = os.environ['FABRIC_DATA_AGENT_ID']
url = f'https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}/dataagents/{agent_id}/agent'

def auth_headers():
    token = notebookutils.credentials.getToken('https://api.fabric.microsoft.com')
    return {'Authorization': f'Bearer {token}'}

async def ask(question):
    async with streamablehttp_client(url, headers=auth_headers()) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = (await session.list_tools()).tools
            if len(tools) != 1:
                raise RuntimeError(f'Expected one agent tool, received {len(tools)}')
            tool = tools[0]
            arg = next(iter(tool.inputSchema['properties']))
            result = await asyncio.wait_for(session.call_tool(tool.name, {arg: question}), timeout=120)
            return '\n'.join(block.text for block in result.content if block.type == 'text')

In [ ]:
questions = [
  'Which routes had the worst average delay and how many passengers were affected?',
  'Compare revenue and load factor by operating company.',
  'Which aircraft have both high downtime and repeated disrupted flights?',
  'Which loyalty tiers lost the most booked revenue to cancellations?'
]
for question in questions:
    print(f'\nQUESTION: {question}\n{await ask(question)}')